# Tutorial 7.2: Application of PRISM to Real Registration-Induced Incomplete in COAD

This tutorial extends the subcellular-resolution COAD adjacent-section analysis from Tutorial 7.1 to a real incomplete-registration setting. Although Xenium RNA and CODEX protein are aligned across serial sections, section offset, tissue loss and non-identical cellular composition leave some CODEX locations without a reliable RNA counterpart.

CODEX protein is treated as the complete source modality. High-confidence RNA transfers define the observed target locations, whereas the remaining CODEX coordinates are RNA-unregistered; their RNA values are genuinely unknown rather than synthetically masked. PRISM therefore tests completion under real cross-section registration uncertainty.


In [ ]:
from pathlib import Path

import scanpy as sc
import PRISM
from PRISM import (compute_similarity_prior, plot_task2_real_three_panel, preprocess_omics,
                   run_clustering_eval_plot, select_best_device, set_prism_plot_style, show_real_missing)
set_prism_plot_style()

In [ ]:
# Load data
DEVICE = select_best_device()
DATASET_DIR = Path("Datasets") / "COAD"
SOURCE_H5AD = DATASET_DIR / "adata_codex.h5ad"
TARGET_H5AD = DATASET_DIR / "adata_reg.h5ad"
REFERENCE_RNA_H5AD = DATASET_DIR / "adata.h5ad"
RESULTS_DIR = Path("Results") / "Tutorial7_2_COAD_incomplete"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PRIOR_PATH = RESULTS_DIR / "COAD_ADT_AOT_32k.npz"
RUN_PREFIX = "COAD_PRISM"

adata_source_raw = sc.read_h5ad(SOURCE_H5AD)
adata_target_raw = sc.read_h5ad(TARGET_H5AD)
adata_reference_raw = sc.read_h5ad(REFERENCE_RNA_H5AD)
adata_source_raw.var_names_make_unique()
adata_target_raw.var_names_make_unique()
adata_reference_raw.var_names_make_unique()


In [ ]:
adata_target_raw.obs['missing']

The registered RNA object preserves CODEX coordinate order: `missing="1"` denotes a high-confidence RNA transfer and `missing="0"` an RNA-unregistered location. The native Xenium RNA section is loaded only for qualitative pattern comparison and does not provide ground truth at the unregistered CODEX coordinates.


In [ ]:
# Display RNA availability after scSLAT confidence filtering
missing_indices, observed_indices = show_real_missing(adata_target_raw, spatial_key="spatial", label_key="missing",
                                                      plot=True, figsize=(4, 4), s=0.01,
                                                      title="COAD RNA: real registration missingness")

print(f"RNA-missing CODEX locations: {len(missing_indices)}/{adata_target_raw.n_obs}")

In [ ]:
# Preprocessing source (CODEX protein) and target (RNA)
adata_source, _ = preprocess_omics(adata_source_raw, modality="ADT", missing_key="missing", 
                                   data_role="source", compute_pca=False, save_raw_eval=False)

adata_target, _ = preprocess_omics(adata_target_raw, modality="RNA", missing_key="missing", min_cells=10,
                                   hvgs=3000, data_role="target", compute_pca=False, save_raw_eval=True)

print("CODEX protein shape after preprocessing:", adata_source.shape)
print("RNA shape after preprocessing:", adata_target.shape)

### Constructing the CODEX similarity prior

Complete CODEX protein profiles define the source spatial-niche prior, which retrieves observed RNA-CODEX locations with related local protein context for RNA-unregistered queries. The 300 AOT environments are constructed in 4,096-cell blocks to keep this large dataset memory-bounded.


In [ ]:
# Compute CODEX similarity prior
# COAD retains 300 AOT environments and uses 4096-cell blocks for memory-bounded AOT.
distance_matrix, _ = compute_similarity_prior(adata_source, adata_target, PRIOR_PATH, device=DEVICE,
                                              covet_k_spatial=32, covet_gene_num=64, aot_k_env=300,
                                              aot_chunk_size=4096, spatial_key="spatial", missing_key="missing",
                                              batch_key="batch" if "batch" in adata_source.obs else -1,
                                              evaluate_prior=False)

In [ ]:
# Constructing spatial graphs
PRISM.Cal_Spatial_Net(adata_source, rad_cutoff=21)
PRISM.Stats_Spatial_Net(adata_source)
PRISM.Cal_Spatial_Net(adata_target, rad_cutoff=21)
PRISM.Stats_Spatial_Net(adata_target)

Radius-based graphs encode local neighbourhoods for the complete CODEX and registered RNA objects in the common CODEX coordinate system.


### Training PRISM

PRISM combines complete CODEX protein, high-confidence RNA-CODEX pairs, spatial graphs and the protein-derived prior to predict RNA at real RNA-unregistered coordinates. The learned representation supports spatial-domain analysis, while the RNA decoder provides the corresponding completion.


In [ ]:
# Train PRISM for real RNA-unregistered CODEX locations
adata_source_out, adata_target_out = PRISM.train_PRISM(adata_source, adata_target, distance_matrix,
                                                       k_top=5, n_epochs=1000, lr=1e-3,
                                                       output_dir=str(RESULTS_DIR), file_prefix=RUN_PREFIX,
                                                       device=DEVICE, patience=20, min_epochs=50,
                                                       center_drop_rate=0.1, noise=0.0,
                                                       interaction_pca=True)

### Task 1: Spatial-domain identification


In [ ]:
# Identify domains from the PRISM embedding and evaluate them post hoc
adata_clustered, domain_metrics = run_clustering_eval_plot(adata_source_out, emb_key="PRISM_emb_base", 
                                                           label_key="spatial_cluster", cluster_key="PRISM_mclust",
                                                           n_clusters=5, s=1, use_pca=True, align_labels=True,
                                                           aligned_key="PRISM_domain", dataset_name="COAD")

### Task 2: Missing-omics imputation

Task 2 provides a qualitative assessment of RNA prediction because true RNA profiles are unavailable at real RNA-unregistered CODEX locations. The three-panel view contrasts the native unaligned RNA section, the confidence-filtered RNA transfer and the PRISM prediction.

In [ ]:
# Compare representative RNA prediction with the native RNA section
fig, axes = plot_task2_real_three_panel(adata_aligned=adata_target_out, prior_matrix=distance_matrix,
                                        save_files=False, output_dir=RESULTS_DIR, file_prefix=RUN_PREFIX,
                                        adata_unaligned_raw=adata_reference_raw, feature="CD68",
                                        show_missing_only=False)